# baselina model for batch monitoring example

In [4]:
# import sys
# print(sys.executable)


In [5]:
# !pip install PyYAML==5.1  # As required by evidently==0.2.0

In [6]:
# !{sys.executable} -m pip show evidently


In [7]:
# !{sys.executable} -m pip uninstall -y evidently
# !pip uninstall -y evidently

# !pip install evidently==0.5.0

# !{sys.executable} -m pip install --upgrade evidently


In [8]:
import requests
import datetime
import pandas as pd
import matplotlib
from joblib import load, dump
from tqdm import tqdm

from evidently import ColumnMapping
from evidently.report import Report
from evidently.metrics import ColumnDriftMetric, DatasetDriftMetric, DatasetMissingValuesMetric


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [9]:
# files = [('green_tripdata_2022-02.parquet', './data'), ('green_tripdata_2022-01.parquet', './data')]
files = [('green_tripdata_2024-03.parquet', './data')]

print("Download files:")
for file, path in files:
    url=f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file}"
    resp=requests.get(url, stream=True)
    save_path=f"{path}/{file}"
    with open(save_path, "wb") as handle:
        for data in tqdm(resp.iter_content(),
                        desc=f"{file}",
                        postfix=f"save to {save_path}",
                        total=int(resp.headers["Content-Length"])):
            handle.write(data)

Download files:


green_tripdata_2024-03.parquet: 100%|█| 1372372/1372372 [00:05<00:00, 230798.53it/s, save to ./data/green_tripdata_2024-


In [10]:
jan_data = pd.read_parquet('data/green_tripdata_2024-03.parquet')

In [11]:
jan_data.describe

<bound method NDFrame.describe of        VendorID lpep_pickup_datetime lpep_dropoff_datetime store_and_fwd_flag  \
0             2  2024-03-01 00:10:52   2024-03-01 00:26:12                  N   
1             2  2024-03-01 00:22:21   2024-03-01 00:35:15                  N   
2             2  2024-03-01 00:45:27   2024-03-01 01:04:32                  N   
3             1  2024-03-01 00:02:00   2024-03-01 00:23:45                  N   
4             2  2024-03-01 00:16:45   2024-03-01 00:23:25                  N   
...         ...                  ...                   ...                ...   
57452         2  2024-03-31 21:19:00   2024-03-31 21:30:00               None   
57453         2  2024-03-31 22:30:00   2024-03-31 22:35:00               None   
57454         2  2024-03-31 22:43:00   2024-03-31 22:48:00               None   
57455         2  2024-03-31 22:48:00   2024-03-31 23:12:00               None   
57456         2  2024-03-31 22:08:00   2024-03-31 22:47:00               No

In [12]:
jan_data.shape

(57457, 20)

In [ ]:
# create target
jan_data["duration_min"] = jan_data.lpep_dropoff_datetime - jan_data.lpep_pickup_datetime
jan_data.duration_min = jan_data.duration_min.apply(lambda td: float(td.total_seconds()) / 60)

In [ ]:
# filter out outliers
jan_data = jan_data[
    (jan_data.duration_min >= 0) & (jan_data.duration_min <= 60)
]
jan_data = jan_data[
    (jan_data.passenger_count > 0) & (jan_data.passenger_count <= 8)
]

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib

In [ ]:
jan_data.duration_min.hist()

In [ ]:
# data labeling
target = "duration_min"
num_features = ["passenger_count", "trip_distance", "fare_amount", "total_amount"]
cat_features = ["PULocationID", "DOLocationID"]

In [ ]:
jan_data.shape

In [ ]:
train_data = jan_data[:30000]
val_data = jan_data[30000:]

In [ ]:
model = LinearRegression()

In [ ]:
model.fit(train_data[num_features + cat_features], train_data[target])

In [ ]:
train_preds = model.predict(train_data[num_features + cat_features])
train_data['prediction'] = train_preds

In [ ]:
val_preds = model.predict(val_data[num_features + cat_features])
val_data['prediction'] = val_preds

In [ ]:
print(mean_absolute_error(train_data.duration_min, train_data.prediction))
print(mean_absolute_error(val_data.duration_min, val_data.prediction))

# Dump model and reference data

In [ ]:
with open('models/lin_reg.bin', 'wb') as f_out:
    dump(model, f_out)

In [ ]:
val_data.to_parquet('data/reference.parquet')

# Evidently Report

In [ ]:
column_mapping = ColumnMapping(
    target=None,
    prediction='prediction',
    numerical_features=num_features,
    categorical_features=cat_features
)

In [ ]:
report = Report(metrics=[
    ColumnDriftMetric(column_name='prediction'),
    DatasetDriftMetric(),
    DatasetMissingValuesMetric()
])

In [ ]:
report.run(reference_data=train_data, current_data=val_data, column_mapping=column_mapping)

In [ ]:
report.show(mode='inline')

In [ ]:
result = report.as_dict()

In [ ]:
result

In [ ]:
# prediction drift
result['metrics'][0]['result']['drift_score']

In [ ]:
# number of drifted columns
result['metrics'][1]['result']

In [ ]:
# share of missing valuse
result['metrics'][2]['result']['current']['share_of_missing_values']

*Evidently Dashboard*

In [ ]:
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.ui.workspace import Workspace
from evidently.ui.dashboards import DashboardPanelCounter, DashboardPanelPlot, CounterAgg, PanelValue, PlotType, ReportFilter
from evidently.renderers.html_widgets import WidgetSize

In [ ]:
ws = Workspace("workspace")

In [ ]:
project = ws.create_project("NYC Taxi Data Quality Project")
project.description = "My project descriotion"
project.save()

In [ ]:
regular_report = Report(
    metrics=[
        DataQualityPreset()
    ],
    timestamp=datetime.datetime(2022, 1, 28)
)

regular_report.run(reference_data=None,
                   current_data=val_data.loc[val_data.lpep_pickup_datetime.between('2022-01-28', '2022-01-29', inclusive="left")],
                   column_mapping=column_mapping)

regular_report

In [ ]:
ws.add_report(project.id, regular_report)

In [3]:
# configure the dashboard
project.dashboard.add_panel(
    DashboardPanelCounter(
        filter=ReportFilter(metadata_values={}, tag_values=[]),
        agg=CounterAgg.NONE,
        title="NYC taxi data dashboard"
    )
)
project.dashboard.add_panel(
    DashboardPanelPlot(
        filter=ReportFilter(metadata_values={}, tag_values=[]),
        title="Inference Count",
        values=[
            PanelValue(
                metric_id="DatasetSummaryMetric",
                field_path="current.number_of_rows",
                legend="count"
            ),
        ],
        plot_type=PlotType.BAR,
        size=WidgetSize.HALF,
    ),
)

project.dashboard.add_panel(
    DashboardPanelPlot(
        filter=ReportFilter(metadata_values={}, tag_values=[]),
        title="number of Missing Values",
        values=[
            PanelValue(
                metric_id="DatasetSummaryMetric",
                field_path="current.number_of_missing_values",
                legend="count"
            ),
        ],
        plot_type=PlotType.LINE,
        size=WidgetSize.HALF,
    ),
)

project.save()

NameError: name 'project' is not defined

In [ ]:
regular_report = Report(
    metrics=[
        DataQualityPreset()
    ],
    timestamp=datetime.datetime(2022, 1, 29)
)

regular_report.run(reference_data=None,
                   current_data=val_data.loc[val_data.lpep_pickup_datetime.between('2022-01-29', '2022-01-30', inclusive="left")],
                   column_mapping=column_mapping)

regular_report

In [ ]:
ws.add_report(project.id, regular_report)